# Research Agent: Tool Use and Reflection

An agentic research pipeline that uses arXiv and web search tools, reflects on the generated report to improve it, and converts the final output to styled HTML.

In [ ]:
import json
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import display, HTML

import research_tools

load_dotenv()
CLIENT = OpenAI()

## Available Research Tools

Two tools are provided via the `research_tools` module:

- **`arxiv_search_tool(query, max_results)`** – retrieves academic papers from the arXiv API.
- **`tavily_search_tool(query, max_results, include_images)`** – runs general web searches via Tavily.

In [ ]:
topic = "linear algebra"
arxiv_results = research_tools.arxiv_search_tool(topic, max_results=3)

for i, paper in enumerate(arxiv_results, 1):
    if "error" in paper:
        print(f"Error: {paper['error']}")
    else:
        print(f"Paper {i}")
        print(f"  Title     : {paper['title']}")
        print(f"  Authors   : {', '.join(paper['authors'])}")
        print(f"  Published : {paper['published']}")
        print(f"  URL       : {paper['url']}\n")

print("\nRaw results:\n")
print(json.dumps(arxiv_results, indent=2))

The `tavily_search_tool` queries the Tavily API and returns a list of dicts with `title`, `content`, and `url` fields (plus optional image URLs when `include_images=True`).

In [ ]:
topic = "retrieval-augmented generation applications"
tavily_results = research_tools.tavily_search_tool(topic)
for item in tavily_results:
    print(item)

In [ ]:
# Tool mapping
TOOL_MAPPING = {
 "tavily_search_tool": research_tools.tavily_search_tool,
 "arxiv_search_tool": research_tools.arxiv_search_tool,
}

## Generating a Research Report with Tools

In [ ]:
def generate_research_report_with_tools(prompt: str, model: str = "gpt-4o") -> str:
    """
    Generates a research report by orchestrating tool calls to arXiv and Tavily,
    then synthesising findings into a structured, cited report.

    Args:
        prompt: Research topic or question.
        model: OpenAI model name.

    Returns:
        Final assistant research report text.
    """
    messages = [
        {
            "role": "system",
            "content": (
                "You are a research assistant that can search the web and arXiv to write detailed, "
                "accurate, and properly sourced research reports.\n\n"
                "Use tools when appropriate (e.g., to find scientific papers or web content).\n"
                "Cite sources whenever relevant. Do NOT omit citations for brevity.\n"
                "When possible, include full URLs (arXiv links, web sources, etc.).\n"
                "Use an academic tone, organise output into clearly labelled sections, and include "
                "inline citations or footnotes as needed.\n"
                "Do not include placeholder text such as '(citation needed)' or '(citations omitted)'."
            ),
        },
        {"role": "user", "content": prompt},
    ]
    tools = [research_tools.arxiv_tool_def, research_tools.tavily_tool_def]
    max_turns = 10

    for _ in range(max_turns):
        response = CLIENT.chat.completions.create(
            model=model,
            messages=messages,
            tools=tools,
            tool_choice="auto",
            temperature=1,
        )
        msg = response.choices[0].message
        messages.append(msg)

        if not msg.tool_calls:
            final_text = msg.content
            print("Final answer:")
            print(final_text)
            break

        for call in msg.tool_calls:
            tool_name = call.function.name
            args = json.loads(call.function.arguments)
            print(f"{tool_name}({args})")
            try:
                result = TOOL_MAPPING[tool_name](**args)
            except Exception as e:
                result = {"error": str(e)}
            messages.append({
                "role": "tool",
                "tool_call_id": call.id,
                "name": tool_name,
                "content": json.dumps(result),
            })

    return final_text

## Reflection and Report Revision

In [ ]:
def reflection_and_rewrite(report, model: str = "gpt-4o-mini", temperature: float = 0.3) -> dict:
    """
    Critiques a research report and produces a revised version.

    Accepts raw text or the messages list from generate_research_report_with_tools.

    Returns:
        dict with keys:
            - "reflection": structured critique of the report
            - "revised_report": improved version of the report
    """
    report = research_tools.parse_input(report)

    user_prompt = f"""Review the following research report and provide:
1. A structured critique identifying strengths, weaknesses, and specific areas for improvement.
2. A fully revised version that addresses all identified issues.

Return ONLY valid JSON with exactly this structure:
{{"reflection": "<structured critique>", "revised_report": "<improved full report>"}}

Report:
{report}"""

    response = CLIENT.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": "You are an academic reviewer and editor."},
            {"role": "user", "content": user_prompt},
        ],
        temperature=temperature,
    )

    llm_output = response.choices[0].message.content.strip()
    try:
        data = json.loads(llm_output)
    except json.JSONDecodeError:
        raise Exception("LLM did not return valid JSON.")

    return {
        "reflection": str(data.get("reflection", "")).strip(),
        "revised_report": str(data.get("revised_report", "")).strip(),
    }

## Converting the Report to HTML

In [ ]:
def convert_report_to_html(report, model: str = "gpt-4o", temperature: float = 0.5) -> str:
    """
    Converts a plaintext research report into a styled HTML document.

    Accepts raw text or the messages list from generate_research_report_with_tools.
    """
    report = research_tools.parse_input(report)
    system_prompt = "You convert plaintext reports into full clean HTML documents."

    user_prompt = f"""Convert the following research report into a complete, styled HTML document with professional CSS.
Return ONLY valid HTML — no markdown, no code fences, no explanations.

Report:
{report}"""

    response = CLIENT.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        temperature=temperature,
    )

    return response.choices[0].message.content.strip()

In [ ]:
prompt_ = "Radio observations of recurrent novae"

# 1) Generate research report using tools
preliminary_report = generate_research_report_with_tools(prompt_)
print("=== Research Report (preliminary) ===\n")
print(preliminary_report)

# 2) Reflect and revise
reflection_text = reflection_and_rewrite(preliminary_report)
print("=== Reflection ===\n")
print(reflection_text['reflection'], "\n")
print("=== Revised Report ===\n")
print(reflection_text['revised_report'], "\n")

# 3) Convert to HTML
html = convert_report_to_html(reflection_text['revised_report'])
print("=== HTML Preview ===\n")
print((html or "")[:600], "\n... [truncated]\n")
display(HTML(html))